In [ ]:
from contextlib import AsyncExitStack
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from ptt_tree_manager import TaskTreeManager,TaskNode
from ptt_reasoning import PTTReasoningModule
from mcp_configure import configure_mcp
from utils import validate_arguments
import json


In [ ]:
load_dotenv()
WORDLIST_PATH = "Imagine_cup_MVP/wordlist.txt"

In [ ]:
stack, GLOBAL_SCHEMA, tools, GLOBAL_NAME_TO_TOOL = await configure_mcp()

llm_for_tool = ChatOpenAI( 
    model="openai/gpt-4.1", 
    openai_api_key=os.getenv("OPEN_AI_API_KEY"), 
    openai_api_base=os.getenv("OPEN_AI_API_BASE")
    ).bind_tools(tools)
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    format="json"
).bind_tools(tools)


goal = "Assess the security posture of example.com"
target = "https://example.com"
constraints = {"scope": "public endpoints only"}

tree_manager = TaskTreeManager()
tree_manager.initialize_tree(goal, target, constraints)
reasoning_module = PTTReasoningModule(tree_manager)

available_tools = list(GLOBAL_NAME_TO_TOOL.keys())


init_prompt = reasoning_module.get_tree_initialization_prompt(goal, target, constraints, available_tools)
init_msg = HumanMessage(content=init_prompt)

init_response = llm.invoke([SystemMessage(content="You are a cybersecurity agent."), init_msg])

parsed_init = reasoning_module.parse_tree_initialization_response(init_response.content)


for task in parsed_init["initial_tasks"]:
    node = TaskNode(
        description=task["description"],
        parent_id=tree_manager.root_id,
        priority=task.get("priority", 5),
        risk_level=task.get("risk_level", "low")
    )
    tree_manager.add_node(node)

#till here code works he hee eehe heeee :) but after that doesnt :( 
while True:

    candidates = tree_manager.get_candidate_tasks()
    if not candidates:
        print("All tasks completed or blocked.")
        break


    next_action_prompt = reasoning_module.get_next_action_prompt(available_tools)

    next_response = llm.invoke([SystemMessage(content="Select next task"), HumanMessage(content=next_action_prompt)])
    print(f"Before Parsing: {next_response}")
    if next_response.tool_calls:
        print(f"Next Response toolcall: \n{json.dumps(next_response.tool_calls[0]["args"],indent=2)}")
        next_action = reasoning_module.parse_next_action_response(json.dumps(next_response.tool_calls[0]["args"],indent=2))
    else:
        next_action = reasoning_module.parse_next_action_response(next_response.content)

    selected_index = next_action.get("selected_task_index", 1) - 1
    selected_task = candidates[selected_index]
    print(f"\n=== Executing Task: {selected_task.description} ===")

    
    tool_name = next_action.get("tool", "manual")
    command = next_action.get("command", "")
    sys_prompt_for_parsing = f"""
You are a STRICT JSON generator.

Your task:
Convert the user's instruction into EXACTLY this JSON schema for the corresponding tool:

Use wordlist path as: {WORDLIST_PATH}

Rules:
- Output ONLY valid JSON
- No markdown
- No explanation
- No extra fields
- All required fields MUST be present
- If information is missing, infer reasonable defaults
"""

    if tool_name in GLOBAL_NAME_TO_TOOL:
        args_by_llm = llm_for_tool.invoke([SystemMessage(content=sys_prompt_for_parsing),HumanMessage(content=command)])
        tool_args = json.loads(args_by_llm.content)
        print(tool_args)
        validation = validate_arguments(tool_args, GLOBAL_SCHEMA[tool_name])
        if validation != "Valid":
            print(f"Argument validation failed: {validation}")
            tool_output = f"Tool execution skipped due to invalid arguments: {validation}"
        else:

            try:
                result = GLOBAL_NAME_TO_TOOL[tool_name].run(tool_args)
                tool_output = result.content[0].text
            except Exception as e:
                print(f"Tool execution error: {e}")
                tool_output = f"Tool execution failed with error: {e}"
    else:
        tool_output = "Manual task, no tool executed."


    update_prompt = reasoning_module.get_tree_update_prompt(tool_output, command, selected_task)
    print(update_prompt)
    update_response = llm.invoke([SystemMessage(content="Update tree based on output"), HumanMessage(content=update_prompt)])


    node_updates, new_tasks = reasoning_module.parse_tree_update_response(update_response.tool_calls[0]['args'])


    tree_manager.update_node(selected_task.id, node_updates)


    for t in new_tasks:
        node = TaskNode(
            description=t["description"],
            parent_id=tree_manager.root_id,
            priority=t.get("priority", 5),
            risk_level=t.get("risk_level", "low")
        )
        tree_manager.add_node(node)


    print(reasoning_module.generate_strategic_summary())

    goal_check_prompt = reasoning_module.get_goal_check_prompt()
    goal_response = llm.invoke([SystemMessage(content="Check goal achievement"), HumanMessage(content=goal_check_prompt)])
    goal_status = reasoning_module.parse_goal_check_response(goal_response.content)
    if goal_status.get("goal_achieved", False):
        print("\n🎯 Goal Achieved!")
        break


await stack.aclose()
